# Notebook 22b — What "Learning From Data" Actually Means
**AI Models and Agents for Business · Week 11 · Session 22**

Today's video defined overfitting and underfitting. This notebook makes them something you can *see and move*, not just define.

**No API key needed** — everything runs locally in Colab. Nothing here calls a language model.

| Part | You'll do |
|---|---|
| A | Watch three models fit the same data — one too simple, one right, one memorizing noise |
| B | Move the dial yourself and find where error actually bottoms out |
| C | See the same effect on a business table, not a curve |

**Callback to Session 5:** Zillow's $304M write-down is this notebook's Part A, at company scale, with real money.

## Cell 1 — Setup

Everything here is preinstalled in Colab. This cell always runs as-is.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures, MinMaxScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

print("Setup complete.")

---
## Part A: Three models, same data

Our dataset: **customer tenure** (months) vs. **monthly spend** (dollars). The true underlying relationship is a saturation curve — spend climbs quickly for new customers, then plateaus. Real data is noisy, so each observed customer sits a little off that true pattern.

The key idea: **a model only ever sees the noisy dots.** It never sees the true curve. Its job is to infer the pattern anyway.

In [ ]:
# The "true" relationship — in real life you never get to see this
def true_pattern(x):
    return 45 + 55 * (1 - np.exp(-x / 7.0))

rng = np.random.RandomState(18)   # fixed so everyone in class sees identical numbers

n_points = 30
x_all = rng.uniform(1, 36, n_points)
y_all = true_pattern(x_all) + rng.normal(0, 9, n_points)

# Split: the model learns on TRAIN and is graded on TEST data it never saw
x_train, x_test, y_train, y_test = train_test_split(
    x_all, y_all, test_size=0.35, random_state=18
)

print(f"Total customers observed : {n_points}")
print(f"Used for training        : {len(x_train)}")
print(f"Held back for testing    : {len(x_test)}")

plt.figure(figsize=(8, 5))
plt.scatter(x_train, y_train, s=60, label="Training data (model sees these)")
plt.scatter(x_test, y_test, s=70, marker="^", label="Test data (held back)")
plt.xlabel("Customer tenure (months)")
plt.ylabel("Monthly spend ($)")
plt.title("The data our model gets to learn from")
plt.legend(); plt.grid(alpha=0.3); plt.show()

## Cell 3 — Fit three models of increasing flexibility

"Degree" is how bendy the model is allowed to be. Degree 1 can only draw a straight line. Degree 15 can wiggle through almost any set of points.

We report **RMSE in dollars** — roughly, "how far off is a typical prediction?" Lower is better.

In [ ]:
def fit_and_score(degree, x_tr, y_tr, x_te, y_te):
    # MinMaxScaler keeps the high-degree math numerically stable
    model = make_pipeline(MinMaxScaler(), PolynomialFeatures(degree), LinearRegression())
    model.fit(x_tr.reshape(-1, 1), y_tr)
    rmse_tr = mean_squared_error(y_tr, model.predict(x_tr.reshape(-1, 1))) ** 0.5
    rmse_te = mean_squared_error(y_te, model.predict(x_te.reshape(-1, 1))) ** 0.5
    return model, rmse_tr, rmse_te

degrees_to_show = [1, 3, 15]
labels = ["UNDERFITTING\n(too simple)", "GOOD FIT", "OVERFITTING\n(memorizing noise)"]

x_smooth = np.linspace(1, 36, 400).reshape(-1, 1)
fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))

for ax, deg, label in zip(axes, degrees_to_show, labels):
    model, tr, te = fit_and_score(deg, x_train, y_train, x_test, y_test)
    ax.scatter(x_train, y_train, s=45, label="Train")
    ax.scatter(x_test, y_test, s=55, marker="^", label="Test")
    ax.plot(x_smooth, model.predict(x_smooth), linewidth=2.5, label=f"Model (deg {deg})")
    ax.plot(x_smooth, true_pattern(x_smooth.ravel()), "--", linewidth=1.5, alpha=0.6, label="True pattern")
    ax.set_title(f"{label}\nTrain RMSE ${tr:,.2f}  |  Test RMSE ${te:,.2f}", fontsize=9)
    ax.set_ylim(0, 140)      # degree 15 shoots far off this chart — that's the point
    ax.set_xlabel("Tenure (months)")
    ax.grid(alpha=0.3); ax.legend(fontsize=7)

axes[0].set_ylabel("Monthly spend ($)")
plt.tight_layout(); plt.show()

print("Notice the degree-15 line leaving the top and bottom of its chart entirely.")

### 💬 Discussion — Part A

The three models score like this:

| Degree | Train RMSE | Test RMSE |
|---|---|---|
| 1 | $10.15 | $12.91 |
| 3 | $9.52 | **$8.62** |
| 15 | **$2.72** | $255,021.49 |

- Degree 15 has by far the **best training error** — it passes almost exactly through the training dots. Why is that a problem rather than a triumph?
- Its test RMSE is about **$255,000** on a quantity that ranges from roughly $45 to $100. In plain language, what is the model doing to produce a number that wrong?
- Which model tracks the dashed "true pattern" most closely? Is it the same model with the lowest training error?
- If a vendor told you their model achieved 99% accuracy, what's the one question you now know to ask? *(Accuracy on what data — data it trained on, or data it had never seen?)*

---
## Part B: Find the sweet spot yourself

Now we sweep every degree from 1 to 15 and plot train error against test error. This produces the single most important chart in applied machine learning.

In [ ]:
degrees = range(1, 16)
train_errors, test_errors = [], []
for d in degrees:
    _, tr, te = fit_and_score(d, x_train, y_train, x_test, y_test)
    train_errors.append(tr); test_errors.append(te)

best_degree = list(degrees)[int(np.argmin(test_errors))]

plt.figure(figsize=(9, 5))
plt.plot(degrees, train_errors, "o-", label="Training error (what the model sees)")
plt.plot(degrees, test_errors, "s-", label="Test error (what actually matters)")
plt.axvline(best_degree, linestyle="--", alpha=0.6, label=f"Best test error: degree {best_degree}")
plt.yscale("log")
plt.xlabel("Model flexibility (polynomial degree)")
plt.ylabel("RMSE in dollars (log scale)")
plt.title("Training error always improves. Test error doesn't.")
plt.legend(); plt.grid(alpha=0.3); plt.show()

print(f"Best degree by test error : {best_degree}")
print(f"Degree 1  -> train ${train_errors[0]:,.2f} | test ${test_errors[0]:,.2f}   (underfit)")
print(f"Degree {best_degree}  -> train ${train_errors[best_degree-1]:,.2f} | test ${test_errors[best_degree-1]:,.2f}   (best)")
print(f"Degree 15 -> train ${train_errors[-1]:,.2f} | test ${test_errors[-1]:,.2f}  (overfit)")

## Cell 5 — 🔬 Your experiment

**✏️ CHANGE ME.** Try these one at a time, re-run, and watch the "best degree" line move:

1. `n_points = 200` — with lots more data, can you get away with more flexibility?
2. `noise_level = 2` — very clean data. Where does the sweet spot go?
3. `noise_level = 30` — very messy data. What happens to the best degree?

The pattern you should find: **more data and cleaner data both let you safely use a more flexible model.** That's not a coincidence — it's the entire reason large AI models require large datasets.

In [ ]:
# ✏️ CHANGE ME — run the three experiments described above
n_points_exp = 30
noise_level = 9

rng2 = np.random.RandomState(18)
x_exp = rng2.uniform(1, 36, n_points_exp)
y_exp = true_pattern(x_exp) + rng2.normal(0, noise_level, n_points_exp)
xtr, xte, ytr, yte = train_test_split(x_exp, y_exp, test_size=0.35, random_state=18)

errs = [fit_and_score(d, xtr, ytr, xte, yte)[2] for d in range(1, 16)]
sweet_spot = int(np.argmin(errs)) + 1
_, tr15, te15 = fit_and_score(15, xtr, ytr, xte, yte)

print(f"Data points: {n_points_exp}   Noise level: {noise_level}")
print(f"Best degree for this data : {sweet_spot}  (test RMSE ${min(errs):,.2f})")
print(f"Degree 15 for comparison  : train ${tr15:,.2f} | test ${te15:,.2f}")

---
## Part C: The same effect on business data, not a curve

Curves are easy to see but nothing like the tables you'll actually work with. Here's the same phenomenon on a customer churn dataset, the kind of table a business actually works with.

A decision tree's flexibility dial is `max_depth`: how many yes/no questions deep it may go. Depth 1 asks one question. Unlimited depth can build rules so specific they effectively memorize individual customers.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

rng3 = np.random.RandomState(7)
n = 400
df = pd.DataFrame({
    "tenure_months":  rng3.randint(1, 60, n),
    "monthly_charge": rng3.uniform(20, 120, n).round(2),
    "support_tickets": rng3.poisson(1.5, n),
})
# True churn drivers: short tenure, high charges, many complaints — plus real randomness
risk = ((df.tenure_months < 12) * 1.4
        + (df.monthly_charge > 85) * 1.0
        + (df.support_tickets >= 3) * 1.2
        + rng3.normal(0, 0.9, n))
df["churned"] = (risk > 1.5).astype(int)

X = df[["tenure_months", "monthly_charge", "support_tickets"]]
y = df["churned"]
Xtr, Xte, ytr2, yte2 = train_test_split(X, y, test_size=0.3, random_state=42)

print(f"{n} customers, {y.mean():.0%} churned\n")
print(f"{'Tree depth':<13}{'Train accuracy':<17}{'Test accuracy':<17}{'Gap'}")
print("-" * 58)
for depth in [1, 2, 3, 5, 10, 20, None]:
    clf = DecisionTreeClassifier(max_depth=depth, random_state=42).fit(Xtr, ytr2)
    a_tr = accuracy_score(ytr2, clf.predict(Xtr))
    a_te = accuracy_score(yte2, clf.predict(Xte))
    shown = "unlimited" if depth is None else str(depth)
    print(f"{shown:<13}{a_tr:<17.1%}{a_te:<17.1%}{a_tr - a_te:+.1%}")

### 💬 Discussion — Part C
- Find the row with the **highest train accuracy**. Is its test accuracy also the highest?
- The unlimited-depth tree hits **100% training accuracy** but drops to about 72% on new customers. In plain business language, what has it actually learned to do?
- Suppose you shipped that unlimited-depth version because it "scored 100% in testing." What does month one in production look like, and how would you discover you had a problem?
- **Callback to Session 5:** Zillow's pricing model performed beautifully on years of rising-price data, then lost $304M when prices fell. Which panel from Part A does that resemble — and was that a *training* failure or a *the world changed* failure? (Compare your answer with the Session 5 case discussion.)

## ✅ Notebook 22b complete — Reflection questions

These prep you for **Quiz 6** (Session 25).

1. Define overfitting in one sentence without using the word "overfit."
2. Why do we hold back test data instead of just measuring performance on everything we have?
3. Training error always falls as a model gets more flexible. Why doesn't test error?
4. Name two things that let you safely use a *more* flexible model.
5. A vendor reports 97% accuracy on their AI product. Write the single diligence question you'd ask.

---
*Next session: the Zillow case, and you train your first real classifier — twice, once with a webcam and once in code.*